# Week 2 — Neural Networks and Deep Learning

This notebook follows the Week 1 education dataset and target (`Exam_Score`). It includes backpropagation from scratch, PyTorch, image transfer learning, embeddings and attention concepts, regularisation, and a fair comparison with the Week 1 baseline.

**Compute note:** XOR and tabular sections run on CPU. The CIFAR-10 transfer-learning section can take time; use a Colab GPU if available. Do not report example scores as your own results.

## 1. A tiny neural network from scratch (NumPy)

Implement a 2–3 neuron hidden layer and solve XOR. The derivative is calculated from the sigmoid output.

In [ ]:
import numpy as np, matplotlib.pyplot as plt

def sigmoid(x): return 1/(1+np.exp(-np.clip(x,-40,40)))
def dsigmoid(y): return y*(1-y)
X_xor=np.array([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
y_xor=np.array([[0.],[1.],[1.],[0.]])
rng=np.random.default_rng(42)
W1=rng.normal(0,1,(2,4)); b1=np.zeros((1,4)); W2=rng.normal(0,1,(4,1)); b2=np.zeros((1,1))
lr=.8; losses=[]
for step in range(20000):
    h=sigmoid(X_xor@W1+b1); out=sigmoid(h@W2+b2)
    loss=np.mean((out-y_xor)**2); losses.append(loss)
    d_out=2*(out-y_xor)/len(y_xor)*dsigmoid(out)
    d_h=(d_out@W2.T)*dsigmoid(h)
    W2-=lr*h.T@d_out; b2-=lr*d_out.sum(axis=0,keepdims=True)
    W1-=lr*X_xor.T@d_h; b1-=lr*d_h.sum(axis=0,keepdims=True)
print('XOR predictions:',out.ravel().round(3),'rounded:',(out.ravel()>.5).astype(int))
plt.plot(losses); plt.xlabel('Training step'); plt.ylabel('MSE'); plt.title('NumPy XOR loss'); plt.show()
assert np.array_equal((out.ravel()>.5).astype(int),y_xor.ravel().astype(int))

**Backpropagation in plain language:** The network first makes a prediction and measures its error. The output error is used to calculate how much each output weight contributed. That responsibility is passed backwards through the hidden layer. Each weight is adjusted a small amount in the direction that reduces the error. Repeating this process gradually improves the predictions.

## 2. Rebuild XOR with PyTorch and train on the Week 1 dataset

`loss.backward()` uses automatic differentiation to calculate gradients; the optimiser applies them. The loop order is zero gradients → forward → loss → backward → step.

In [ ]:
import os, time, random
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',device)
X_t=torch.tensor(X_xor,dtype=torch.float32,device=device); y_t=torch.tensor(y_xor,dtype=torch.float32,device=device)
xor_net=nn.Sequential(nn.Linear(2,8),nn.Tanh(),nn.Linear(8,1),nn.Sigmoid()).to(device)
loss_fn=nn.BCELoss(); opt=torch.optim.Adam(xor_net.parameters(),lr=.03)
for epoch in range(3000):
    opt.zero_grad(); pred=xor_net(X_t); loss=loss_fn(pred,y_t); loss.backward(); opt.step()
print('PyTorch XOR:',xor_net(X_t).detach().cpu().numpy().round(3).ravel())

In [ ]:
DATA_URL='https://raw.githubusercontent.com/omaarrx/Student-Score-Prediction/main/StudentPerformanceFactors.csv'
if not os.path.exists('StudentPerformanceFactors.csv'):
    import urllib.request; urllib.request.urlretrieve(DATA_URL,'StudentPerformanceFactors.csv')
df=pd.read_csv('StudentPerformanceFactors.csv').dropna().copy()
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from sklearn.ensemble import RandomForestRegressor,HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
TARGET='Exam_Score'; excluded=['Gender','Learning_Disabilities']
X=df.drop(columns=[TARGET]+excluded,errors='ignore'); y=df[TARGET].astype('float32')
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42)
num=X.select_dtypes(include=np.number).columns.tolist(); cat=X.select_dtypes(exclude=np.number).columns.tolist()
prep=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore'))]),cat)])
Xtr_arr=prep.fit_transform(Xtr); Xte_arr=prep.transform(Xte)
if hasattr(Xtr_arr,'toarray'): Xtr_arr=Xtr_arr.toarray(); Xte_arr=Xte_arr.toarray()
Xtr_t=torch.tensor(Xtr_arr,dtype=torch.float32); Xte_t=torch.tensor(Xte_arr,dtype=torch.float32)
ytr_t=torch.tensor(ytr.to_numpy().reshape(-1,1),dtype=torch.float32); yte_t=torch.tensor(yte.to_numpy().reshape(-1,1),dtype=torch.float32)
class TabularNet(nn.Module):
    def __init__(self,n):
        super().__init__(); self.net=nn.Sequential(nn.Linear(n,64),nn.ReLU(),nn.Linear(64,32),nn.ReLU(),nn.Linear(32,1))
    def forward(self,x): return self.net(x)
net=TabularNet(Xtr_t.shape[1]).to(device); opt=torch.optim.Adam(net.parameters(),lr=1e-3,weight_decay=1e-4); mse=nn.MSELoss()
train_ds=TensorDataset(Xtr_t,ytr_t); loader=DataLoader(train_ds,batch_size=128,shuffle=True)
start=time.perf_counter(); losses=[]
for epoch in range(80):
    net.train(); running=0
    for xb,yb in loader:
        xb,yb=xb.to(device),yb.to(device); opt.zero_grad(); pred=net(xb); l=mse(pred,yb); l.backward(); opt.step(); running+=l.item()*len(xb)
    losses.append(running/len(train_ds))
train_seconds=time.perf_counter()-start
net.eval()
with torch.no_grad(): pred_nn=net(Xte_t.to(device)).cpu().numpy().ravel()
print('NN MAE/RMSE/R2:',mean_absolute_error(yte,pred_nn),mean_squared_error(yte,pred_nn)**.5,r2_score(yte,pred_nn),'train sec:',round(train_seconds,2))
plt.plot(losses); plt.xlabel('Epoch'); plt.ylabel('Training MSE'); plt.title('PyTorch training loss'); plt.show()

### CPU vs GPU timing

The dataset is small. CPU may be faster because moving small batches to a GPU has overhead. The measurement below uses the same prepared tensors and one full forward/backward step.

In [ ]:
def timing_step(device_name):
    model=TabularNet(Xtr_t.shape[1]).to(device_name); xx=Xtr_t.to(device_name); yy=ytr_t.to(device_name)
    optim=torch.optim.Adam(model.parameters(),lr=1e-3)
    if device_name.type=='cuda': torch.cuda.synchronize()
    t0=time.perf_counter()
    for _ in range(20):
        optim.zero_grad(); loss=mse(model(xx),yy); loss.backward(); optim.step()
    if device_name.type=='cuda': torch.cuda.synchronize()
    return time.perf_counter()-t0
cpu_time=timing_step(torch.device('cpu')); gpu_time=timing_step(device) if torch.cuda.is_available() else None
print({'CPU_20_steps_seconds':cpu_time,'GPU_20_steps_seconds':gpu_time})

## 3. CNN and transfer learning

Use CIFAR-10 (10 classes, 50,000 training images). Train a small CNN first, then compare with an ImageNet-pretrained ResNet18 whose feature layers are frozen. This dataset is a generic computer-vision exercise alongside the education tabular project. For a faster run, reduce `MAX_TRAIN` and `EPOCHS`; record the exact settings.

In [ ]:
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import Subset
MEAN=(.485,.456,.406); STD=(.229,.224,.225)
train_tf=transforms.Compose([transforms.Resize(224),transforms.RandomHorizontalFlip(),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
test_tf=transforms.Compose([transforms.Resize(224),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
train_ds_img=datasets.CIFAR10('data',train=True,download=True,transform=train_tf)
test_ds_img=datasets.CIFAR10('data',train=False,download=True,transform=test_tf)
MAX_TRAIN=5000; MAX_TEST=1000; EPOCHS=2
train_ds_img=Subset(train_ds_img,range(MAX_TRAIN)); test_ds_img=Subset(test_ds_img,range(MAX_TEST))
train_loader=DataLoader(train_ds_img,batch_size=64,shuffle=True,num_workers=2,pin_memory=torch.cuda.is_available())
test_loader=DataLoader(test_ds_img,batch_size=64,shuffle=False,num_workers=2)
classes=train_ds_img.dataset.classes
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__(); self.features=nn.Sequential(nn.Conv2d(3,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(64,128,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1)); self.fc=nn.Linear(128,10)
    def forward(self,x): return self.fc(self.features(x).flatten(1))
def train_image_model(model,optimizer,epochs=EPOCHS):
    model.to(device); criterion=nn.CrossEntropyLoss()
    for ep in range(epochs):
        model.train(); total=correct=0
        for xb,yb in train_loader:
            xb,yb=xb.to(device),yb.to(device); optimizer.zero_grad(); logits=model(xb); loss=criterion(logits,yb); loss.backward(); optimizer.step(); total+=len(yb); correct+=(logits.argmax(1)==yb).sum().item()
        print(f'epoch {ep+1}: train accuracy {correct/total:.3f}')
    model.eval(); ys=[]; ps=[]
    with torch.no_grad():
        for xb,yb in test_loader:
            logits=model(xb.to(device)); ps.extend(logits.argmax(1).cpu().tolist()); ys.extend(yb.tolist())
    from sklearn.metrics import accuracy_score,confusion_matrix
from sklearn.tree import DecisionTreeClassifier
    return accuracy_score(ys,ps),np.array(ys),np.array(ps)
cnn=SmallCNN(); cnn_acc,cnn_y,cnn_p=train_image_model(cnn,torch.optim.Adam(cnn.parameters(),lr=1e-3)); print('Small CNN test accuracy:',cnn_acc)

In [ ]:
weights=models.ResNet18_Weights.DEFAULT
resnet=models.resnet18(weights=weights)
for p in resnet.parameters(): p.requires_grad=False
resnet.fc=nn.Linear(resnet.fc.in_features,10)
resnet_acc,res_y,res_p=train_image_model(resnet,torch.optim.Adam(resnet.fc.parameters(),lr=1e-3))
print('Frozen ResNet18 test accuracy:',resnet_acc)
# Fine-tune last residual block at a lower learning rate
for p in resnet.layer4.parameters(): p.requires_grad=True
optimizer=torch.optim.Adam([{'params':resnet.layer4.parameters(),'lr':1e-4},{'params':resnet.fc.parameters(),'lr':1e-3}])
ft_acc,ft_y,ft_p=train_image_model(resnet,optimizer,epochs=1)
print('Fine-tuned ResNet18 test accuracy:',ft_acc)
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_predictions(res_y,res_p,display_labels=classes,xticks_rotation=90); plt.title('ResNet18 confusion matrix'); plt.tight_layout(); plt.show()
wrong=np.where(res_y!=res_p)[0][:5]
print('First five misclassified test examples:')
for idx in wrong: print('true=',classes[res_y[idx]],'predicted=',classes[res_p[idx]])

## 4. Embeddings, semantic search and attention

Embed ten domain-specific sentences, compare cosine similarities and retrieve the top three for a query. An embedding model is a useful semantic representation, but can still fail on domain-specific meaning.

In [ ]:
!pip -q install sentence-transformers seaborn

In [ ]:
from sentence_transformers import SentenceTransformer
import seaborn as sns
sentences=[
 'Regular attendance can help students keep up with lessons.',
 'Students who study consistently may improve their exam preparation.',
 'Sleep supports memory and concentration during learning.',
 'Tutoring can provide additional explanations for difficult topics.',
 'Parental encouragement may support study routines.',
 'Physical activity is associated with general wellbeing.',
 'A school library provides access to learning materials.',
 'High absenteeism can make it harder to follow classroom instruction.',
 'Revision before an examination helps recall key concepts.',
 'A balanced routine includes rest, study and recreation.'
]
embedder=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
emb=embedder.encode(sentences,normalize_embeddings=True)
sim=emb@emb.T
plt.figure(figsize=(9,7)); sns.heatmap(sim,xticklabels=range(1,11),yticklabels=range(1,11),cmap='viridis'); plt.title('Sentence cosine similarity'); plt.xlabel('Sentence'); plt.ylabel('Sentence'); plt.show()
query='How can a learner avoid missing lessons?'
q=embedder.encode([query],normalize_embeddings=True)[0]
hits=np.argsort(emb@q)[::-1][:3]
print([(round(float(emb[i]@q),3),sentences[i]) for i in hits])
print('Inspect similarity scores for one false match or domain-specific failure and record why the model may misunderstand it.')

### RNNs, attention and transformers — written answers

- **Why RNNs struggle with long sentences:** information must pass through many sequential steps; early details can fade and training has difficulty preserving long-range dependencies.
- **What attention adds:** each token can directly weigh information from other tokens, including distant ones.
- **Self-attention:** tokens in a sequence compute relationships with other tokens in that same sequence to build context-aware representations.
- **Why transformers parallelise:** self-attention processes token positions together during training rather than requiring a recurrent step for each token. (Autoregressive generation is still sequential across newly generated tokens.)

Draw a transformer block and label token embedding, positional information, multi-head self-attention, feed-forward layer, residual connections, layer normalisation and output head. Positional information is required because attention alone does not encode token order.

## 5. Regularisation and experiment tracking

Demonstrate overfitting on a small neural network and compare dropout, weight decay, early stopping and data augmentation. For this tabular exercise, augmentation is represented by adding small noise to scaled continuous training features only; it is a simple experiment, not a universally valid method.

In [ ]:
from sklearn.metrics import mean_squared_error
# Train/validation split from training portion
Xtr2,Xval,ytr2,yval=train_test_split(Xtr_arr,ytr.to_numpy(),test_size=.2,random_state=42)
class RegNet(nn.Module):
    def __init__(self,drop=0.0):
        super().__init__(); self.net=nn.Sequential(nn.Linear(Xtr2.shape[1],128),nn.ReLU(),nn.Dropout(drop),nn.Linear(128,64),nn.ReLU(),nn.Linear(64,1))
    def forward(self,x): return self.net(x)
def fit_variant(drop=0.,wd=0.,early=True,noise=0.,epochs=150):
    torch.manual_seed(SEED)
    model=RegNet(drop).to(device); optimizer=torch.optim.Adam(model.parameters(),lr=1e-3,weight_decay=wd); criterion=nn.MSELoss()
    xt=torch.tensor(Xtr2,dtype=torch.float32); yt=torch.tensor(ytr2.reshape(-1,1),dtype=torch.float32)
    xv=torch.tensor(Xval,dtype=torch.float32,device=device); yv=torch.tensor(yval.reshape(-1,1),dtype=torch.float32,device=device)
    best=float('inf'); best_state=None; wait=0; hist=[]
    for ep in range(epochs):
        model.train(); order=torch.randperm(len(xt))
        for ix in order.split(128):
            xb=xt[ix].to(device); yb=yt[ix].to(device)
            if noise: xb=xb+noise*torch.randn_like(xb)
            optimizer.zero_grad(); loss=criterion(model(xb),yb); loss.backward(); optimizer.step()
        model.eval()
        with torch.no_grad():
            tr=criterion(model(xt.to(device)),yt.to(device)).item(); val=criterion(model(xv),yv).item()
        hist.append((tr,val))
        if val<best-1e-4: best=val; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; wait=0
        else: wait+=1
        if early and wait>=12: break
    model.load_state_dict(best_state)
    return model,np.array(hist),best
variants={'baseline':dict(early=False),'dropout':dict(drop=.3,early=True),'weight_decay':dict(wd=1e-3,early=True),'early_stopping':dict(early=True),'noise_aug':dict(noise=.02,early=True)}
exp={}
for name,kwargs in variants.items():
    m,h,b=fit_variant(**kwargs); exp[name]=(m,h,b); print(name,'epochs=',len(h),'best val MSE=',round(b,3))
plt.figure(figsize=(8,4))
for name,(m,h,b) in exp.items(): plt.plot(h[:,1],label=name)
plt.xlabel('Epoch'); plt.ylabel('Validation MSE'); plt.title('Regularisation experiments'); plt.legend(); plt.show()


In [ ]:
print('Experiment table:')
for name,(m,h,b) in exp.items(): print({'technique':name,'best_validation_MSE':round(b,3),'epochs':len(h)})
print('Combine the two best validation techniques in a follow-up run and note whether the combination improves on either alone.')

## 6. Compare deep learning against Week 1 baseline

Use exactly the same held-out test rows and metrics. Include linear regression, random forest, histogram gradient boosting and the PyTorch neural network. Tabular data often favours tree ensembles; an honest negative result for deep learning is acceptable.

In [ ]:
comparison=[]
for name,est in [('Linear regression',LinearRegression()),('Random forest',RandomForestRegressor(n_estimators=200,min_samples_leaf=2,random_state=42,n_jobs=-1)),('HistGradientBoosting',HistGradientBoostingRegressor(random_state=42))]:
    p=Pipeline([('prep',prep),('model',est)]); t0=time.perf_counter(); p.fit(Xtr,ytr); fit_sec=time.perf_counter()-t0; pr=p.predict(Xte)
    comparison.append({'model':name,'MAE':mean_absolute_error(yte,pr),'RMSE':mean_squared_error(yte,pr)**.5,'R2':r2_score(yte,pr),'train_seconds':fit_sec,'parameters_or_size':'tree/linear model'})
comparison.append({'model':'PyTorch neural network','MAE':mean_absolute_error(yte,pred_nn),'RMSE':mean_squared_error(yte,pred_nn)**.5,'R2':r2_score(yte,pred_nn),'train_seconds':train_seconds,'parameters_or_size':sum(p.numel() for p in net.parameters())})
comparison_df=pd.DataFrame(comparison).sort_values('RMSE'); display(comparison_df.round(3))
print('Conclusion prompt: Was the neural network worth its training and implementation cost for this dataset? Explain using the actual table, not a pre-written claim.')

## Final checklist

- [ ] XOR learned using NumPy backpropagation and PyTorch autograd
- [ ] Same Week 1 target predicted with a PyTorch model
- [ ] CPU/GPU timing recorded with hardware details
- [ ] CNN and pretrained ResNet transfer learning compared; confusion matrix and five errors inspected
- [ ] Ten sentences embedded; heatmap and top-3 semantic search shown; a failure case discussed
- [ ] Transformer diagram added by hand
- [ ] Four regularisation methods compared; experiment log committed
- [ ] Week 1 models and neural network compared on same test split

**Caution:** CIFAR-10 is a generic vision benchmark; it does not replace an image dataset from the education domain. The main domain comparison is the tabular Exam_Score task. Do not fabricate performance metrics or mark checklist items complete until you have run the notebook.